In [ ]:
pip install mysqlx-connector-python

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import mysql.connector
from mysql.connector import Error


def connect_to_database():
    try:
        conn = mysql.connector.connect(
            host='localhost',
            user='formuser',               # Replace with your MySQL username
            password='YourStrongPassword', # Replace with your MySQL password
            database='registration_db'     # Replace with your database name
        )
        if conn.is_connected():
            return conn
    except Error as e:
        messagebox.showerror("Database Error", f"Connection Failed:\n{e}")
        return None


def submit_form():
    name = entry_name.get()
    email = entry_email.get()
    phone = entry_phone.get()
    password = entry_password.get()
    confirm_password = entry_confirm_password.get()
    gender = gender_var.get()
    country = country_var.get()

    # Validation
    if not (name and email and phone and password and confirm_password and gender and country):
        messagebox.showerror("Error", "All fields are required!")
        return
    if password != confirm_password:
        messagebox.showerror("Error", "Passwords do not match!")
        return

    # Database Insertion
    conn = connect_to_database()
    if conn:
        try:
            cursor = conn.cursor()
            query = """
            INSERT INTO users (name, email, phone, password, gender, country)
            VALUES (%s, %s, %s, %s, %s, %s)
            """
            values = (name, email, phone, password, gender, country)
            cursor.execute(query, values)
            conn.commit()
            conn.close()
            messagebox.showinfo("Success", "Registration Successful!")

            # Clear the form
            entry_name.delete(0, tk.END)
            entry_email.delete(0, tk.END)
            entry_phone.delete(0, tk.END)
            entry_password.delete(0, tk.END)
            entry_confirm_password.delete(0, tk.END)
            gender_var.set(None)
            country_var.set("Select Country")

        except Error as e:
            messagebox.showerror("Error", f"Database error:\n{e}")
            conn.rollback()
            conn.close()


root = tk.Tk()
root.title("User Registration Form")
root.geometry("400x500")
root.resizable(False, False)

tk.Label(root, text="Full Name:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
entry_name = tk.Entry(root, font=("Arial", 12))
entry_name.pack(fill="x", padx=20)

tk.Label(root, text="Email:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
entry_email = tk.Entry(root, font=("Arial", 12))
entry_email.pack(fill="x", padx=20)

tk.Label(root, text="Phone Number:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
entry_phone = tk.Entry(root, font=("Arial", 12))
entry_phone.pack(fill="x", padx=20)

tk.Label(root, text="Password:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
entry_password = tk.Entry(root, font=("Arial", 12), show="*")
entry_password.pack(fill="x", padx=20)

tk.Label(root, text="Confirm Password:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
entry_confirm_password = tk.Entry(root, font=("Arial", 12), show="*")
entry_confirm_password.pack(fill="x", padx=20)

gender_var = tk.StringVar()
tk.Label(root, text="Gender:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
frame_gender = tk.Frame(root)
frame_gender.pack(anchor="w", padx=20)
tk.Radiobutton(frame_gender, text="Male", variable=gender_var, value="Male").pack(side="left", padx=5)
tk.Radiobutton(frame_gender, text="Female", variable=gender_var, value="Female").pack(side="left", padx=5)

tk.Label(root, text="Country:", font=("Arial", 12)).pack(pady=5, anchor="w", padx=20)
country_var = tk.StringVar()
countries = ["Select Country", "USA", "UK", "Germany", "Pakistan", "India"]
country_dropdown = ttk.Combobox(root, textvariable=country_var, values=countries, state="readonly")
country_dropdown.current(0)
country_dropdown.pack(fill="x", padx=20)

tk.Button(root, text="Register", font=("Arial", 14), bg="blue", fg="white", command=submit_form).pack(pady=20)

root.mainloop()
